# Simulation 1b: Refinement nach ersten Ergebnissen

**Erkenntnisse aus Sim 1:**
- Energieminimierung funktioniert, Fixpunkt ist tot
- Signalmetriken liefern echten Mehrwert gegenueber Delta
- Orthogonalitaet ist hilfreich aber nicht zwingend
- Diversity-Repulsion zerstoert Konvergenz (zu aggressiv)
- Kein Worker-Kollaps bei orthogonalen Raeumen

**Offene Punkte fuer 1b:**
1. Constraint-basierte Diversity statt Gegenkraft
2. Energie + Diversity + Signalmetriken kombiniert
3. Learning Rate Sweep
4. Signalmetriken bei Diversity-Szenarien

Autoren: Toby Brummer & Claude | Datum: 2026-03-29

In [1]:
# 0. Setup
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
sys.path.insert(0, '.')

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product
import warnings
warnings.filterwarnings('ignore')

from sim1_helpers import (
    generate_orthogonal_subspaces, load_semantic_vectors,
    merge_average, merge_phase_alignment, MERGE_STRATEGIES,
    iterate_fixpoint, iterate_energy,
)
from sim1_metrics import (
    compute_snr, compute_phase_coherence, compute_crest_factor,
    compute_all_metrics, halting_decision, compare_halting_methods,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
sns.set_theme(style="whitegrid", palette="muted")

# --- Neue Funktionen fuer 1b ---

def mean_pairwise_cosine(states):
    """Mittlere paarweise Cosine Similarity."""
    flat = [s.flatten() for s in states]
    sims = []
    for i in range(len(flat)):
        for j in range(i + 1, len(flat)):
            ni, nj = np.linalg.norm(flat[i]), np.linalg.norm(flat[j])
            if ni > 1e-10 and nj > 1e-10:
                sims.append(np.dot(flat[i], flat[j]) / (ni * nj))
    return np.mean(sims) if sims else 0.0

def iterate_energy_v2(
    worker_states, subspaces, merge_fn,
    max_iter=50, epsilon=1e-6, lr=0.1,
    diversity_weight=0.1, max_cosine=0.8,
):
    """
    Energieminimierung v2: Diversity als Soft-Constraint.
    Penalty nur aktiv wenn paarweise Cosine > max_cosine.
    """
    import torch.nn.functional as F
    
    history = {"states": [], "deltas": [], "energies": [], "merged": [],
               "diversity_penalties": []}
    
    states_t = [torch.tensor(s.flatten(), dtype=torch.float32, requires_grad=True)
                for s in worker_states]
    subspaces_t = [torch.tensor(P, dtype=torch.float32) for P in subspaces]

    max_delta = 0.0
    for t in range(max_iter):
        history["states"].append([s.detach().numpy().copy() for s in states_t])

        # Konsensus-Energie: Paarweise Distanzen
        energy = torch.tensor(0.0)
        for i in range(len(states_t)):
            for j in range(i + 1, len(states_t)):
                dist = torch.norm(states_t[i] - states_t[j])
                energy = energy + dist ** 2

        # Diversity-Penalty: Nur aktiv wenn Cosine > max_cosine
        div_penalty = torch.tensor(0.0)
        for i in range(len(states_t)):
            for j in range(i + 1, len(states_t)):
                cos_sim = F.cosine_similarity(
                    states_t[i].unsqueeze(0), states_t[j].unsqueeze(0)
                ).squeeze()
                # Quadratische Penalty nur ueber Schwelle
                excess = torch.clamp(cos_sim - max_cosine, min=0.0)
                div_penalty = div_penalty + excess ** 2
        
        total_energy = energy + diversity_weight * div_penalty
        history["energies"].append(energy.item())
        history["diversity_penalties"].append(div_penalty.item())

        total_energy.backward()

        new_states = []
        for i, (s, P) in enumerate(zip(states_t, subspaces_t)):
            with torch.no_grad():
                grad = s.grad if s.grad is not None else torch.zeros_like(s)
                new_s = s - lr * grad
                projected = P @ (P.T @ new_s)
                new_states.append(projected)
            s.grad = None

        deltas = [torch.norm(n - o).item() for n, o in zip(new_states, states_t)]
        max_delta = max(deltas)
        history["deltas"].append(max_delta)

        merged = merge_fn([s.detach().numpy() for s in new_states])
        history["merged"].append(merged.copy())

        states_t = [s.clone().detach().requires_grad_(True) for s in new_states]

        if max_delta < epsilon:
            break

    history["states"].append([s.detach().numpy().copy() for s in states_t])
    history["n_iterations"] = t + 1
    history["converged"] = max_delta < epsilon
    return history

print("Setup complete. iterate_energy_v2 loaded.")

Setup complete. iterate_energy_v2 loaded.


## Experiment G: Constraint-basierte Diversity

Sim 1 Experiment E zeigte: Repulsive Kraft zerstört Konvergenz.
Neuer Ansatz: Penalty nur aktiv wenn paarweise Cosine Similarity über Schwelle.

Sweep über diversity_weight und max_cosine.

In [3]:
# Experiment G: Constraint-basierte Diversity
# Sweep: diversity_weight × max_cosine

dim_g = 256
n_sub_g = 5
subspace_dim_g = dim_g // (n_sub_g * 2)

diversity_weights = [0.0, 0.1, 1.0, 10.0]
max_cosines = [0.9, 0.7, 0.5, 0.3]

results_g = []

for dw in diversity_weights:
    for mc in max_cosines:
        np.random.seed(SEED)
        torch.manual_seed(SEED)
        
        subspaces = generate_orthogonal_subspaces(n_sub_g, dim_g, subspace_dim_g)
        workers = [P @ np.random.randn(subspace_dim_g) for P in subspaces]
        
        history = iterate_energy_v2(
            workers, subspaces, merge_average,
            max_iter=50, lr=0.1,
            diversity_weight=dw, max_cosine=mc,
        )
        
        cos_final = mean_pairwise_cosine(history["states"][-1])
        metrics = compute_all_metrics(history)
        
        results_g.append({
            "diversity_weight": dw,
            "max_cosine": mc,
            "converged": history["converged"],
            "iterations": history["n_iterations"],
            "final_cosine": cos_final,
            "final_snr": metrics["snr"][-1] if metrics["snr"] else None,
            "div_penalty_final": history["diversity_penalties"][-1] if history["diversity_penalties"] else 0,
        })

# Tabelle
print(f"{'dw':>6} {'mc':>6} {'conv':>6} {'iter':>5} {'cos':>8} {'snr':>8} {'penalty':>8}")
print("-" * 55)
for r in results_g:
    snr_s = f"{r['final_snr']:.1f}" if r['final_snr'] and abs(r['final_snr']) < 1000 else "inf"
    print(f"{r['diversity_weight']:>6.1f} {r['max_cosine']:>6.1f} "
          f"{str(r['converged']):>6} {r['iterations']:>5} "
          f"{r['final_cosine']:>8.3f} {snr_s:>8} {r['div_penalty_final']:>8.4f}")

    dw     mc   conv  iter      cos      snr  penalty
-------------------------------------------------------
   0.0    0.9   True    11    0.000      inf   0.0000
   0.0    0.7   True    11    0.000      inf   0.0000
   0.0    0.5   True    11    0.000      inf   0.0000
   0.0    0.3   True    11    0.000      inf   0.0000
   0.1    0.9   True    11    0.000      inf   0.0000
   0.1    0.7   True    11    0.000      inf   0.0000
   0.1    0.5   True    11    0.000      inf   0.0000
   0.1    0.3   True    11    0.000      inf   0.0000
   1.0    0.9   True    11    0.000      inf   0.0000
   1.0    0.7   True    11    0.000      inf   0.0000
   1.0    0.5   True    11    0.000      inf   0.0000
   1.0    0.3   True    11    0.000      inf   0.0000
  10.0    0.9   True    11    0.000      inf   0.0000
  10.0    0.7   True    11    0.000      inf   0.0000
  10.0    0.5   True    11    0.000      inf   0.0000
  10.0    0.3   True    11    0.000      inf   0.0000


## Experiment H: Learning Rate Sweep

Sim 1 nutzte lr=0.1 überall. Wie sensitiv ist die Konvergenz?

In [4]:
# Experiment H: Learning Rate Sweep

dim_h = 256
n_sub_h = 5
subspace_dim_h = dim_h // (n_sub_h * 2)
learning_rates = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]

results_h = []

for lr in learning_rates:
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    
    subspaces = generate_orthogonal_subspaces(n_sub_h, dim_h, subspace_dim_h)
    workers = [P @ np.random.randn(subspace_dim_h) for P in subspaces]
    
    history = iterate_energy(workers, subspaces, merge_average, max_iter=100, lr=lr)
    metrics = compute_all_metrics(history)
    
    results_h.append({
        "lr": lr,
        "converged": history["converged"],
        "iterations": history["n_iterations"],
        "final_snr": metrics["snr"][-1] if metrics["snr"] else None,
        "snr_curve": metrics["snr"],
        "delta_curve": history["deltas"],
    })

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Experiment H: Learning Rate Sweep (Energie, dim=256, n=5)", fontsize=14)

for r in results_h:
    deltas = r["delta_curve"]
    ax1.plot(deltas, label=f"lr={r['lr']}", marker='o', markersize=2)
    snr = [s for s in r["snr_curve"] if abs(s) < 1000]
    if snr:
        ax2.plot(snr, label=f"lr={r['lr']}", marker='o', markersize=2)

ax1.set_yscale('log')
ax1.set_title("Delta über Iterationen")
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Delta (log)")
ax1.legend(fontsize=8)

ax2.set_title("SNR über Iterationen")
ax2.set_xlabel("Iteration")
ax2.set_ylabel("SNR (dB)")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig("exp_h_learning_rate.png", dpi=150)
plt.show()

print(f"{'lr':>6} {'conv':>6} {'iter':>5}")
print("-" * 20)
for r in results_h:
    print(f"{r['lr']:>6.2f} {str(r['converged']):>6} {r['iterations']:>5}")

LinAlgError: SVD did not converge

## Experiment I: Signalmetriken bei Diversity-Konvergenz

Der entscheidende Test: Funktionieren die Signalmetriken auch als Diagnostik wenn Diversity-Constraints aktiv sind? Kann man den Trade-off zwischen Konsensus und Diversity über SNR/Coherence/Crest Factor ablesen?

In [ ]:
# Experiment I: Signalmetriken bei Diversity-Szenarien
# Vergleich: Energie ohne Diversity vs. mit Diversity-Constraint

dim_i = 256
n_sub_i = 5
subspace_dim_i = dim_i // (n_sub_i * 2)

configs_i = [
    {"name": "No diversity", "dw": 0.0, "mc": 0.8},
    {"name": "Soft (dw=0.1, mc=0.8)", "dw": 0.1, "mc": 0.8},
    {"name": "Medium (dw=1.0, mc=0.5)", "dw": 1.0, "mc": 0.5},
    {"name": "Strong (dw=10.0, mc=0.3)", "dw": 10.0, "mc": 0.3},
]

fig, axes = plt.subplots(len(configs_i), 4, figsize=(20, 4 * len(configs_i)))
fig.suptitle("Experiment I: Signalmetriken mit Diversity-Constraints", fontsize=14)
col_labels = ["SNR (dB)", "Phase Coherence", "Crest Factor", "Cosine Sim"]

for row, cfg in enumerate(configs_i):
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    
    subspaces = generate_orthogonal_subspaces(n_sub_i, dim_i, subspace_dim_i)
    workers = [P @ np.random.randn(subspace_dim_i) for P in subspaces]
    
    history = iterate_energy_v2(
        workers, subspaces, merge_average,
        max_iter=50, lr=0.1,
        diversity_weight=cfg["dw"], max_cosine=cfg["mc"],
    )
    metrics = compute_all_metrics(history)
    cos_per_iter = [mean_pairwise_cosine(states) for states in history["states"]]
    
    # SNR
    ax = axes[row, 0]
    snr_vals = [s for s in metrics["snr"] if abs(s) < 1000]
    ax.plot(snr_vals, 'b-o', markersize=3)
    ax.set_ylabel(cfg["name"], fontsize=9)
    if row == 0: ax.set_title(col_labels[0])
    
    # Phase Coherence
    ax = axes[row, 1]
    coh_matrix = np.array([c[:3] for c in metrics["coherence"]])
    for k in range(min(3, coh_matrix.shape[1])):
        ax.plot(coh_matrix[:, k], marker='o', markersize=3, label=f'C{k+1}')
    ax.set_ylim(-0.1, 1.1)
    ax.legend(fontsize=7)
    if row == 0: ax.set_title(col_labels[1])
    
    # Crest Factor
    ax = axes[row, 2]
    ax.plot(metrics["crest_factor"], 'r-o', markersize=3)
    ax.axhline(y=3.0, color='gray', linestyle=':', alpha=0.5)
    if row == 0: ax.set_title(col_labels[2])
    
    # Cosine Similarity
    ax = axes[row, 3]
    ax.plot(cos_per_iter, 'purple', marker='o', markersize=3)
    ax.axhline(y=cfg["mc"], color='orange', linestyle='--', alpha=0.7, label=f'max_cos={cfg["mc"]}')
    ax.set_ylim(-0.5, 1.1)
    ax.legend(fontsize=7)
    if row == 0: ax.set_title(col_labels[3])
    
    print(f"{cfg['name']}: converged={history['converged']}, iters={history['n_iterations']}, "
          f"final_cos={cos_per_iter[-1]:.3f}, final_snr={metrics['snr'][-1]:.1f} dB"
          if abs(metrics['snr'][-1]) < 1000
          else f"{cfg['name']}: converged={history['converged']}, iters={history['n_iterations']}, "
               f"final_cos={cos_per_iter[-1]:.3f}, final_snr=inf")

plt.tight_layout()
plt.savefig("exp_i_diversity_signals.png", dpi=150)
plt.show()

## Zusammenfassung 1b

In [ ]:
# Zusammenfassung 1b

print("=" * 60)
print("SIMULATION 1b: ERGEBNIS-ZUSAMMENFASSUNG")
print("=" * 60)
print()

# Aus G: Konvergiert Energie mit Diversity-Constraint?
g_converged = [r for r in results_g if r["converged"] and r["diversity_weight"] > 0]
g_best = min(g_converged, key=lambda r: r["iterations"]) if g_converged else None

print("Experiment G (Constraint-Diversity):")
if g_best:
    print(f"  Bestes Setup: dw={g_best['diversity_weight']}, mc={g_best['max_cosine']}")
    print(f"  Konvergiert in {g_best['iterations']} Iter, final_cosine={g_best['final_cosine']:.3f}")
else:
    print("  Kein konvergentes Setup mit Diversity gefunden")

print()
print("Experiment H (Learning Rate):")
for r in results_h:
    if r["converged"]:
        print(f"  lr={r['lr']}: {r['iterations']} Iterationen")

print()
print("Gesamtbilanz Simulation 1 + 1b:")
print("  - Energieminimierung: funktioniert (Sim 1)")
print("  - Signalmetriken > Delta: bestaetigt (Sim 1, Exp D)")
print("  - Orthogonalitaet: hilfreich, nicht zwingend (Sim 1, Exp B)")
if g_best:
    print(f"  - Diversity-Constraint: funktioniert mit dw={g_best['diversity_weight']}, mc={g_best['max_cosine']}")
else:
    print("  - Diversity-Constraint: weitere Arbeit noetig")
print()
print("NAECHSTER SCHRITT: Simulation 2 (Probing an bestehendem Modell)")